In [1]:
import pandas as pd
import numpy as np

from scipy.stats import wasserstein_distance

In [2]:
REFERENCE_CSV = (
    "../../data/processed/CTB/sampled_real_eventlogs/s6_sample_24.000_eventlog_target_rank_features.csv"
)

SIM_CSV = (
    "../../data/processed/CTB/prosit_simulations/sim_log_s6_sample_30.000_depth10.csv"
)

ref = pd.read_csv(REFERENCE_CSV)
sim = pd.read_csv(SIM_CSV)

print(
    f"Reference events: {len(ref):,}"
)

print(
    f"Simulation events: {len(sim):,}"
)

Reference events: 74,066
Simulation events: 75,158


In [ ]:
# =============CALCULATE SIM KPIS=============================================
# CALCULATE SIMULATION KPIS
# ==========================================================

for col in [
    "enabled:timestamp",
    "start:timestamp",
    "time:timestamp"
]:
    sim[col] = pd.to_datetime(sim[col])

# ----------------------------------------------------------
# EVENT LEVEL KPIs
# ----------------------------------------------------------

sim["waiting_time"] = (
    sim["start:timestamp"]
    - sim["enabled:timestamp"]
).dt.total_seconds() / 60

sim["service_time"] = (
    sim["time:timestamp"]
    - sim["start:timestamp"]
).dt.total_seconds() / 60

# ----------------------------------------------------------
# CASE LEVEL KPI
# ----------------------------------------------------------

case_start = (
    sim.groupby(
        "case:concept:name"
    )["start:timestamp"]
    .transform("min")
)

case_end = (
    sim.groupby(
        "case:concept:name"
    )["time:timestamp"]
    .transform("max")
)

sim["turnaround_time"] = (
    case_end
    - case_start
).dt.total_seconds() / 60

# ----------------------------------------------------------
# RMG EVENTS ONLY
# ----------------------------------------------------------

rmg_activities = [
    "RMG_receive",
    "RMG_delivery",
    "RMG_mixed"
]

sim_rmg = sim[
    sim["concept:name"]
    .isin(rmg_activities)
].copy()

ref_rmg = ref[
    ref["concept:name"]
    .isin(rmg_activities)
].copy()

print(
    f"Reference RMG events: {len(ref_rmg):,}"
)

print(
    f"Simulation RMG events: {len(sim_rmg):,}"
)

Reference RMG events: 24,000
Simulation RMG events: 25,331


In [4]:
#========== CALCULTATE REF KPIS =================
# ==========================================================
# CALCULATE REFERENCE KPIS
# ==========================================================

for col in [
    "enabled:timestamp",
    "start:timestamp",
    "time:timestamp"
]:
    ref[col] = pd.to_datetime(ref[col])

# ----------------------------------------------------------
# EVENT LEVEL KPIs
# ----------------------------------------------------------

ref["waiting_time"] = (
    ref["start:timestamp"]
    - ref["enabled:timestamp"]
).dt.total_seconds() / 60

ref["service_time"] = (
    ref["time:timestamp"]
    - ref["start:timestamp"]
).dt.total_seconds() / 60

# ----------------------------------------------------------
# CASE LEVEL KPI
# ----------------------------------------------------------

ref_case_start = (
    ref.groupby(
        "case:concept:name"
    )["start:timestamp"]
    .transform("min")
)

ref_case_end = (
    ref.groupby(
        "case:concept:name"
    )["time:timestamp"]
    .transform("max")
)

ref["turnaround_time"] = (
    ref_case_end
    - ref_case_start
).dt.total_seconds() / 60

# ----------------------------------------------------------
# RMG EVENTS ONLY
# ----------------------------------------------------------

rmg_activities = [
    "RMG_receive",
    "RMG_delivery",
    "RMG_mixed"
]

ref_rmg = ref[
    ref["concept:name"]
    .isin(rmg_activities)
].copy()

print(
    f"Reference RMG events: {len(ref_rmg):,}"
)

print(
    f"Reference cases: "
    f"{ref['case:concept:name'].nunique():,}"
)

print(
    "\nKPI columns created:"
)

print(
    [
        "waiting_time",
        "service_time",
        "turnaround_time"
    ]
)

Reference RMG events: 24,000
Reference cases: 24,000

KPI columns created:
['waiting_time', 'service_time', 'turnaround_time']


In [5]:
# ==========================================================
# WAITING TIME BY ACTIVITY
# ==========================================================

activity_waiting = pd.DataFrame({
    "ref_mean_waiting":
        ref.groupby(
            "concept:name"
        )["waiting_time"]
        .mean(),

    "sim_mean_waiting":
        sim.groupby(
            "concept:name"
        )["waiting_time"]
        .mean()
})

activity_waiting["difference"] = (
    activity_waiting["sim_mean_waiting"]
    - activity_waiting["ref_mean_waiting"]
)

activity_waiting.sort_values(
    "difference",
    ascending=False
)

,ref_mean_waiting,sim_mean_waiting,difference
concept:name,,,
HO2_receive,13.007843,136.070300,123.062456
HO2_delivery,13.656250,131.071318,117.415068
HO2_mixed,7.476923,119.354659,111.877735
RMG_delivery,6.166911,30.930623,24.763712
RMG_receive,7.623269,29.861101,22.237832
RMG_mixed,7.989294,26.490984,18.501690
Gate Out,8.360875,13.861153,5.500278
LL_delivery,2.450912,6.666667,4.215755
LL_mixed,2.581283,6.735725,4.154441


In [6]:
# ==========================================================
# WAITING TIME BY RESOURCE
# ==========================================================

resource_waiting = pd.DataFrame({

    "ref_mean_waiting":
        ref.groupby(
            "org:resource"
        )["waiting_time"]
        .mean(),

    "sim_mean_waiting":
        sim.groupby(
            "org:resource"
        )["waiting_time"]
        .mean()

})

resource_waiting["difference"] = (
    resource_waiting["sim_mean_waiting"]
    - resource_waiting["ref_mean_waiting"]
)

resource_waiting.sort_values(
    "difference",
    ascending=False
)

,ref_mean_waiting,sim_mean_waiting,difference
org:resource,,,
HO2,12.560000,130.634048,118.074048
T07,7.733112,60.263070,52.529958
T08,10.349943,59.361520,49.011576
T27,5.308118,47.770675,42.462557
T09,7.406348,46.055590,38.649243
T11,8.709797,46.987834,38.278037
T06,8.893617,45.843970,36.950353
T23,7.057143,40.992060,33.934918
T14,6.137040,37.358946,31.221906


In [8]:
# ==========================================================
# RESOURCE FREQUENCY
# ==========================================================

ref_res = (
    ref["org:resource"]
    .value_counts()
    .rename("reference")
)

sim_res = (
    sim["org:resource"]
    .value_counts()
    .rename("simulation")
)

resource_load = pd.concat(
    [ref_res, sim_res],
    axis=1
).fillna(0)

resource_load["difference"] = (
    resource_load["simulation"]
    - resource_load["reference"]
)

resource_load.sort_values(
    "difference",
    ascending=False
)

,reference,simulation,difference
HO2,250,1571,1321
T15,932,1312,380
T06,1081,1362,281
T17,1257,1454,197
T26,783,963,180
T18,977,1150,173
T14,1412,1574,162
T12,1183,1327,144
T13,1529,1657,128
T24,1292,1402,110


In [10]:
# ==========================================================
# TOTAL WAITING CONTRIBUTION
# ==========================================================

ref_total_waiting = (
    ref.groupby(
        "org:resource"
    )["waiting_time"]
    .sum()
)

sim_total_waiting = (
    sim.groupby(
        "org:resource"
    )["waiting_time"]
    .sum()
)

waiting_contribution = pd.DataFrame({

    "ref_waiting":
        ref_total_waiting,

    "sim_waiting":
        sim_total_waiting

})

waiting_contribution["extra_waiting"] = (
    waiting_contribution["sim_waiting"]
    - waiting_contribution["ref_waiting"]
)

waiting_contribution.sort_values(
    "extra_waiting",
    ascending=False
)

,ref_waiting,sim_waiting,extra_waiting
org:resource,,,
HO2,3140.0,205226.089792,202086.089792
Res.GateOut,200661.0,332667.674456,132006.674456
T06,9614.0,62439.487439,52825.487439
T14,8665.5,58802.981159,50137.481159
T27,5754.0,55222.900422,49468.900422
T11,9424.0,52391.434843,42967.434843
T07,6983.0,48391.245229,41408.245229
T09,7117.5,47437.258211,40319.758211
T08,9139.0,47845.384843,38706.384843


In [11]:
waiting_contribution = pd.DataFrame({

    "ref_total":
    ref.groupby("org:resource")
       ["waiting_time"].sum(),

    "sim_total":
    sim.groupby("org:resource")
       ["waiting_time"].sum()
})

waiting_contribution["extra_waiting"] = (
    waiting_contribution["sim_total"]
    - waiting_contribution["ref_total"]
)

waiting_contribution.sort_values(
    "extra_waiting",
    ascending=False
).head(10)

,ref_total,sim_total,extra_waiting
org:resource,,,
HO2,3140.0,205226.089792,202086.089792
Res.GateOut,200661.0,332667.674456,132006.674456
T06,9614.0,62439.487439,52825.487439
T14,8665.5,58802.981159,50137.481159
T27,5754.0,55222.900422,49468.900422
T11,9424.0,52391.434843,42967.434843
T07,6983.0,48391.245229,41408.245229
T09,7117.5,47437.258211,40319.758211
T08,9139.0,47845.384843,38706.384843
